# Fase 2 — Modelos ML: Regresión Logística y Random Forest
Implementación, búsqueda de hiperparámetros y evaluación final de los modelos de machine learning. Se combina vectorización TF-IDF con 7 variables lingüísticas mediante ColumnTransformer. Se utiliza GridSearchCV con Stratified K-Fold para evitar data leakage y garantizar representación balanceada de clases en cada fold (Bhattacharjee et al., 2023; Anan et al., 2023; Fatima et al., 2024).

Prompt utilizado para la implementación del Pipeline:
*'Tengo un dataset de detección de sarcasmo en español con desbalance de clases 2:1. Quiero implementar Regresión Logística y Random Forest con TF-IDF. Necesito hacer búsqueda de hiperparámetros con GridSearchCV y Stratified K-Fold de 5 folds optimizando F1-Macro. ¿Cómo evito data leakage al incluir TF-IDF en el cross-validation? Genera el código correcto en Python con scikit-learn.'*

## 1. Importación de librerías

In [25]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report
import joblib

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 2. Carga de datos preprocesados

In [26]:
df_train = pd.read_csv('../data/train_clean.csv')
df_test  = pd.read_csv('../data/test_clean.csv')

FEATURE_COLS = ['n_exc', 'n_int', 'n_may', 'n_emo', 'n_ris', 'n_neg']

X_train = df_train[['MESSAGE_CLEAN'] + FEATURE_COLS]
y_train = df_train['IS_IRONIC']
X_test  = df_test[['MESSAGE_CLEAN'] + FEATURE_COLS]
y_test  = df_test['IS_IRONIC']

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Distribución train: {y_train.value_counts().to_dict()}")

Train: (7197, 7) | Test: (1800, 7)
Distribución train: {0: 4797, 1: 2400}


## 3. Definición de pipelines y grillas de hiperparámetros
Se utiliza ColumnTransformer para combinar TF-IDF sobre el texto con los 7 features lingüísticos numéricos. El ColumnTransformer se incluye dentro del Pipeline para evitar data leakage: el TF-IDF se reajusta únicamente con los datos de entrenamiento de cada fold.

Se eliminan stopwords del español dado que TF-IDF no captura contexto gramatical, a diferencia de los modelos DL del estudio (Manoleasa et al., 2022). Se aplica class_weight='balanced' para compensar el desbalance 2:1 del dataset.

In [27]:
stop_words_es = stopwords.words('spanish')

# ColumnTransformer: TF-IDF sobre texto + passthrough para features lingüísticos
preprocessor = ColumnTransformer([
    ('tfidf', TfidfVectorizer(
        stop_words=stop_words_es,
        max_features=10000  # Pandey & Singh (2023); Šandor & Bagić Babac (2024)
    ), 'MESSAGE_CLEAN'),
    ('ling', 'passthrough', FEATURE_COLS)
])

# Pipeline LR
pipeline_lr = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

# Pipeline RF
pipeline_rf = Pipeline([
    ('prep', ColumnTransformer([
        ('tfidf', TfidfVectorizer(stop_words=stop_words_es, max_features=10000), 'MESSAGE_CLEAN'),
        ('ling', 'passthrough', FEATURE_COLS)
    ])),
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# Grillas de hiperparámetros
param_grid_lr = {
    'prep__tfidf__ngram_range': [(1,1), (1,2), (2,3)],
    'clf__C':                   [0.1, 1.0, 10],
    'clf__solver':              ['lbfgs', 'liblinear']
}

param_grid_rf = {
    'prep__tfidf__ngram_range':     [(1,1), (1,2), (2,3)],
    'clf__n_estimators':            [100, 200],       # Pandey & Singh (2023)
    'clf__max_depth':               [None, 10, 20],   # Pandey & Singh (2023)
    'clf__min_samples_split':       [2, 5]            # Pandey & Singh (2023)
}

print("Pipelines y grillas definidos")

Pipelines y grillas definidos


## 4. Búsqueda de hiperparámetros — Regresión Logística
StratifiedKFold con 5 folds garantiza la misma proporción de clases en cada partición, necesario ante el desbalance 2:1 del dataset. La métrica de optimización es F1-Macro, consistente con la métrica principal del estudio (Bhattacharjee et al., 2023; Manoleasa et al., 2022).

In [28]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Buscando hiperparámetros para LR...")
search_lr = GridSearchCV(
    pipeline_lr, param_grid_lr,
    cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1
)
search_lr.fit(X_train, y_train)
print(f"Mejores parámetros LR: {search_lr.best_params_}")
print(f"Mejor F1-Macro LR (CV): {search_lr.best_score_:.4f}")

Buscando hiperparámetros para LR...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Mejores parámetros LR: {'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf__ngram_range': (1, 2)}
Mejor F1-Macro LR (CV): 0.6590


## 5. Búsqueda de hiperparámetros — Random Forest

In [29]:
print("Buscando hiperparámetros para RF...")
search_rf = GridSearchCV(
    pipeline_rf, param_grid_rf,
    cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1
)
search_rf.fit(X_train, y_train)
print(f"Mejores parámetros RF: {search_rf.best_params_}")
print(f"Mejor F1-Macro RF (CV): {search_rf.best_score_:.4f}")

Buscando hiperparámetros para RF...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Mejores parámetros RF: {'clf__max_depth': None, 'clf__min_samples_split': 5, 'clf__n_estimators': 100, 'prep__tfidf__ngram_range': (1, 2)}
Mejor F1-Macro RF (CV): 0.6395


## 6. Evaluación final en conjunto de test

In [30]:
best_lr = search_lr.best_estimator_
best_rf = search_rf.best_estimator_

y_pred_lr = best_lr.predict(X_test)
y_pred_rf = best_rf.predict(X_test)

print("=== Regresión Logística ===")
print(f"F1-Macro test: {f1_score(y_test, y_pred_lr, average='macro'):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=['No irónico', 'Irónico']))

print("\n=== Random Forest ===")
print(f"F1-Macro test: {f1_score(y_test, y_pred_rf, average='macro'):.4f}")
print(classification_report(y_test, y_pred_rf, target_names=['No irónico', 'Irónico']))

=== Regresión Logística ===
F1-Macro test: 0.6653
              precision    recall  f1-score   support

  No irónico       0.78      0.76      0.77      1201
     Irónico       0.55      0.57      0.56       599

    accuracy                           0.70      1800
   macro avg       0.66      0.67      0.67      1800
weighted avg       0.70      0.70      0.70      1800


=== Random Forest ===
F1-Macro test: 0.6363
              precision    recall  f1-score   support

  No irónico       0.74      0.84      0.79      1201
     Irónico       0.57      0.42      0.48       599

    accuracy                           0.70      1800
   macro avg       0.66      0.63      0.64      1800
weighted avg       0.69      0.70      0.69      1800



## 7. Exportación de modelos

In [31]:
joblib.dump(best_lr, '../data/modelo_lr.pkl')
joblib.dump(best_rf, '../data/modelo_rf.pkl')
print("Modelos guardados: modelo_lr.pkl | modelo_rf.pkl")

Modelos guardados: modelo_lr.pkl | modelo_rf.pkl
